# Cost optimisation algorithms

**Data sources:** 

* production - solarEdge (ID:1567917)
* consumption - mvm (GroupID: 4)
* price - HUPX.hu



Using cassandra to store the aforementioned sources after transformations: 
* **consumer_profile**
*  **solar_panel_production**
*  **hupx_energy_price**

Data is stored in 15 minute time intervals. production and consumption is stored in **kwh**, price is in **mwh**



In [2]:
spark.version

'3.5.0'

In [1]:
import findspark
import itertools
import pyspark.sql
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, abs, weekofyear, to_timestamp, col, to_utc_timestamp, count, max, monotonically_increasing_id, lit, concat, expr, regexp_replace,lpad
 

findspark.init()

spark = SparkSession.builder \
    .appName("CassandraConncection") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0")\
    .config("spark.sql.catalog.client","com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.sql.catalog.client.spark.cassandra.connection.host","127.0.0.1")\
    .getOrCreate()
    #.config("spark.cassandra.connection.port", "9042")\
    


24/06/07 12:12:14 WARN Utils: Your hostname, PySpark resolves to a loopback address: 127.0.1.1; using 192.168.64.129 instead (on interface ens33)
24/06/07 12:12:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/student/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/student/.ivy2/cache
The jars for the packages stored in: /home/student/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1fc00419-7ba0-4df8-a003-3f840e7f3a86;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.0 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found com.datastax.oss#java-driver-core-shaded;4.13.0 in central
	found com.datastax.oss#native-protocol;1.5.0 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3

# Extract


In [24]:

#reading and transforming csv input, only run this block when needed.
consumption = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .load("../ingest/consumption_profile.csv")

production = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",",")\
          .load("../ingest/production_profile.csv")

price = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .option("inferSchema","true")\
          .load("../ingest/price.csv")

consumption.show()
production.show()
price.show()






+----------+----------+--------+
|     Dátum|Negyedórák| Group#4|
+----------+----------+--------+
|2022.01.01|      0:00|0.018103|
|2022.01.01|      0:15|0.018038|
|2022.01.01|      0:30|0.018010|
|2022.01.01|      0:45|0.018095|
|2022.01.01|      1:00|0.018187|
|2022.01.01|      1:15|0.018158|
|2022.01.01|      1:30|0.018087|
|2022.01.01|      1:45|0.018103|
|2022.01.01|      2:00|0.018172|
|2022.01.01|      2:15|0.018167|
|2022.01.01|      2:30|0.018129|
|2022.01.01|      2:45|0.018056|
|2022.01.01|      3:00|0.017923|
|2022.01.01|      3:15|0.017764|
|2022.01.01|      3:30|0.017725|
|2022.01.01|      3:45|0.017770|
|2022.01.01|      4:00|0.017912|
|2022.01.01|      4:15|0.018011|
|2022.01.01|      4:30|0.018170|
|2022.01.01|      4:45|0.018460|
+----------+----------+--------+
only showing top 20 rows

+----------------+--------------+
|            time|production_(W)|
+----------------+--------------+
|01.01.2022 00:00|             0|
|01.01.2022 00:15|             0|
|01.01.2022 

# Transform

Transformations are needed: All timezones should be casted to **UTC**, granuality should be set to 15 minutes to all sources

production profile:

In [26]:
#yearly 50 kw
spark.conf.set("spark.sql.session.timeZone", "UTC") # necessary config to aviod clock shift
production_final = production.withColumn("timestamp",to_timestamp(col("time"),"dd.MM.yyyy HH:mm"))\
                        .withColumn("profile_id", lit(1567917))\
                        .withColumn("production_(W)", col("production_(W)")*50/10*0.00025)\
                        .withColumnRenamed("production_(W)", "production_kwh")\
                        .select("profile_id","timestamp","production_kwh")
                        
#num = production_final.toPandas()["production_kwh"].sum()/50/2
#print(num)


consumption profile:

In [27]:
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
cons_final = consumption.withColumn("time", to_timestamp(concat(col("Dátum"), lit(" "), col("Negyedórák")), "yyyy.MM.dd HH:mm"))\
                        .withColumn("utc_time", to_utc_timestamp(col("time"), "Europe/Budapest"))\
                        .withColumn("consumption_kwh", col("Group#4").cast("double")*10)\
                        .withColumn("rownum", monotonically_increasing_id())

cons_filter = cons_final.groupBy("utc_time").agg(count("*").alias("count")).filter(col("count")>=2)
cons_filtered = cons_final.join(cons_filter,"utc_time","inner").groupBy("utc_time").agg(max("rownum").alias("rownum"))\
                          .withColumn("modified_utc_time", expr("utc_time + INTERVAL 1 HOUR"))\
                          .select("modified_utc_time","rownum")
cons_final = cons_final.join(cons_filtered,"rownum","left_outer")\
                       .withColumn("timestamp", expr("IFNULL(modified_utc_time, utc_time)"))\
                       .withColumn("profile_id", lit(4))\
                       .select("profile_id","timestamp","consumption_kwh")

cons_final.show()

+----------+-------------------+-------------------+
|profile_id|          timestamp|    consumption_kwh|
+----------+-------------------+-------------------+
|         4|2021-12-31 23:00:00|0.18103000000000002|
|         4|2021-12-31 23:15:00|0.18037999999999998|
|         4|2021-12-31 23:30:00|             0.1801|
|         4|2021-12-31 23:45:00|            0.18095|
|         4|2022-01-01 00:00:00|0.18186999999999998|
|         4|2022-01-01 00:15:00|0.18158000000000002|
|         4|2022-01-01 00:30:00|0.18086999999999998|
|         4|2022-01-01 00:45:00|0.18103000000000002|
|         4|2022-01-01 01:00:00|            0.18172|
|         4|2022-01-01 01:15:00|            0.18167|
|         4|2022-01-01 01:30:00|            0.18129|
|         4|2022-01-01 01:45:00|            0.18056|
|         4|2022-01-01 02:00:00|            0.17923|
|         4|2022-01-01 02:15:00|            0.17764|
|         4|2022-01-01 02:30:00|0.17725000000000002|
|         4|2022-01-01 02:45:00|0.177700000000

price:

In [28]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
price_final = price.withColumn("Hours",regexp_replace("Hours","[HB]",""))\
                   .withColumn("Hours", expr("cast(Hours as int) - 1"))\
                   .withColumn("Quarters",expr("explode(array(':00',':15',':30',':45'))"))\
                   .withColumn("Hours", concat(lpad("Hours",2,"0"),"Quarters"))\
                   .withColumn("time",to_timestamp(concat("Delivery day",lit(" "),"Hours"),"dd.MM.yyyy HH:mm"))\
                   .withColumn("utc_timestamp", to_utc_timestamp("time", "Europe/Budapest"))\
                   .withColumn("rownum", monotonically_increasing_id())

price_filter = price_final.groupBy("utc_timestamp").agg(count("*").alias("count")).filter(col("count")>=2)
price_filtered = price_final.join(price_filter,"utc_timestamp","inner").groupBy("utc_timestamp").agg(max("rownum").alias("rownum"))\
                            .withColumn("modified_utc_timestamp", expr("utc_timestamp + INTERVAL 1 HOUR")).select("modified_utc_timestamp","rownum")
price_final = price_final.join(price_filtered,"rownum","left_outer")\
                         .withColumn("timestamp", expr("IFNULL(modified_utc_timestamp, utc_timestamp)"))\
                         .withColumnRenamed("Prices (EUR/Mwh)","price_eur_per_mwh")\
                         .select("timestamp","price_eur_per_mwh")

                   

price_final.show()
price_final.printSchema()
price_final.count()

+-------------------+-----------------+
|          timestamp|price_eur_per_mwh|
+-------------------+-----------------+
|2021-12-31 23:00:00|            61.84|
|2021-12-31 23:15:00|            61.84|
|2021-12-31 23:30:00|            61.84|
|2021-12-31 23:45:00|            61.84|
|2022-01-01 00:00:00|            41.33|
|2022-01-01 00:15:00|            41.33|
|2022-01-01 00:30:00|            41.33|
|2022-01-01 00:45:00|            41.33|
|2022-01-01 01:00:00|            43.22|
|2022-01-01 01:15:00|            43.22|
|2022-01-01 01:30:00|            43.22|
|2022-01-01 01:45:00|            43.22|
|2022-01-01 02:00:00|            45.46|
|2022-01-01 02:15:00|            45.46|
|2022-01-01 02:30:00|            45.46|
|2022-01-01 02:45:00|            45.46|
|2022-01-01 03:00:00|            37.67|
|2022-01-01 03:15:00|            37.67|
|2022-01-01 03:30:00|            37.67|
|2022-01-01 03:45:00|            37.67|
+-------------------+-----------------+
only showing top 20 rows

root
 |-- time

70080

# Load

In [30]:
cons_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="consumption_profile", keyspace="energycom") \
    .save()

In [9]:
production_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="production_profile", keyspace="energycom") \
    .save()

In [10]:
price_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .save()

# Using csv data (optional)

In [29]:
consumption = cons_final
production = production_final
price = price_final

# Reading from Cassandra DB


In [6]:
consumption = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="consumption_profile", keyspace="energycom") \
    .load()
production = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="production_profile", keyspace="energycom") \
    .load()
price = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .load()

consumption.show()
production.show()
price.show()

+-------------------+----------+---------------+
|          timestamp|profile_id|consumption_kwh|
+-------------------+----------+---------------+
|2022-04-02 05:00:00|         4|       0.015565|
|2023-03-29 20:45:00|         4|       0.016705|
|2023-12-20 13:45:00|         4|        0.05586|
|2023-09-12 17:30:00|         4|       0.018821|
|2022-02-14 17:00:00|         4|       0.037702|
|2022-01-28 18:45:00|         4|       0.024338|
|2022-02-09 01:45:00|         4|       0.017825|
|2022-08-09 20:30:00|         4|        0.01213|
|2023-12-04 13:30:00|         4|       0.055788|
|2023-10-09 18:45:00|         4|       0.015885|
|2023-08-17 20:00:00|         4|       0.012694|
|2022-05-19 08:45:00|         4|       0.063398|
|2023-11-24 13:00:00|         4|       0.059866|
|2022-04-14 23:30:00|         4|       0.014147|
|2023-09-04 22:45:00|         4|       0.012529|
|2022-11-29 13:00:00|         4|       0.058106|
|2022-02-28 10:15:00|         4|       0.083938|
|2023-07-30 04:30:00

# Combining data

In [30]:
from pyspark.sql.functions import *

# filter
consumption   = consumption.filter(col("profile_id") == 4)
production = production.filter(col("profile_id") == 1567917)

# join
process = consumption.select("timestamp","consumption_kwh").join(production.select("timestamp","production_kwh"),"timestamp","inner")\
                  .join(price.select("timestamp","price_eur_per_mwh"),"timestamp","inner")

#sort and cast to Pandas
process = process.orderBy("timestamp")\
                 .withColumn("consumption_kwh", process["consumption_kwh"].cast("float"))\
                 .withColumn("production_kwh", process["production_kwh"].cast("float"))\
                 .withColumn("price_eur_per_mwh", process["price_eur_per_mwh"].cast("float"))\
                 .toPandas()
process.head()

,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
0,2022-01-01 00:00:00,0.18187,0.0,41.330002
1,2022-01-01 00:15:00,0.18158,0.0,41.330002
2,2022-01-01 00:30:00,0.18087,0.0,41.330002
3,2022-01-01 00:45:00,0.18103,0.0,41.330002
4,2022-01-01 01:00:00,0.18172,0.0,43.220001


# Optimistaion strategies

Goal: minimise **energy costs**

Scenarios:
* pv only
* greedy strategy
* rule based strategy full
* rule based strategy without LT

# PV only

There is no BESS in this scenario, if there is a sufficit, we sell it to the grid, if there is a power deficit, we buy from the grid.

In [31]:
pv = process.assign(feeding_grid = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                    taking_from_grid   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                    sell_price = lambda x: x["feeding_grid"]*x["price_eur_per_mwh"]*0.00025,
                    buy_price = lambda x: x["taking_from_grid"]*x["price_eur_per_mwh"]*0.00025,
                    net_revenue = lambda x: x["sell_price"] - x["buy_price"])

pv.head()

,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,feeding_grid,taking_from_grid,sell_price,buy_price,net_revenue
0,2022-01-01 00:00:00,0.18187,0.0,41.330002,0.0,0.18187,0.0,0.001879,-0.001879
1,2022-01-01 00:15:00,0.18158,0.0,41.330002,0.0,0.18158,0.0,0.001876,-0.001876
2,2022-01-01 00:30:00,0.18087,0.0,41.330002,0.0,0.18087,0.0,0.001869,-0.001869
3,2022-01-01 00:45:00,0.18103,0.0,41.330002,0.0,0.18103,0.0,0.001870,-0.001870
4,2022-01-01 01:00:00,0.18172,0.0,43.220001,0.0,0.18172,0.0,0.001963,-0.001963


In [32]:
pv_total_generation = pv.assign(generation = lambda x: x['production_kwh'])['generation'].sum()
pv_total_export = pv["feeding_grid"].sum()
pv_self_consumption_rate = 1 - pv_total_export/pv_total_generation

pv_total_consumption = pv.assign(consumption = lambda x: x['consumption_kwh'])['consumption'].sum()
pv_total_import = pv['taking_from_grid'].sum()
pv_self_sufficiency_rate = 1 - pv_total_import/pv_total_consumption

total_cost = pv['buy_price'].sum()
total_revenue = pv['sell_price'].sum()
net_revenue = pv['net_revenue'].sum()

print(f"Total generation: {pv_total_generation}")
print(f"Total consumption: {pv_total_consumption}")
print(f"Self consumption rate: {pv_self_consumption_rate}")
print(f"Self sufficiency_rate: {pv_self_sufficiency_rate}")
print(f"Total cost: {total_cost}")
print(f"Total revenue: {total_revenue}")
print(f"net revenue: {net_revenue}")


Total generation: 142054.3125
Total consumption: 19999.2734375
Self consumption rate: 0.07790863513946533
Self sufficiency_rate: 0.5533823668956757
Total cost: 438.0835876464844
Total revenue: 6176.375
net revenue: 5738.2919921875


# Greedy strategy

If production covers consumption, then surplus is stored into BESS, if it is full, then the remainder is sold to the grid.
If production does not cover consumption, then the deficit is first covered from BESS, and if the energy stored in BESS was not sufficient, then power is bought from the grid.



In [17]:
Capacity = 100
Charge_rate = 50
Discharge_rate = 50
Capacity_min = 0


In [33]:
import numpy as np
# calculating energy surplus and deficit from the difference of production and consumption,
# SOC colums are added for the beggining and end of 15 minute intervals, setting default value to 0. 
basic_p = process.assign(energy_to_store = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                       energy_to_get   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                       battery_charge_start = float(0.00000),
                       battery_charge_end   = float(0.00000))
basic_p.head()

#from pyspark.sql.functions import when
#basic_p = process.withColumn("energy_to_store", when(col("production_kwh")-col("consumption_kwh")>0,col("production_kwh")-col("consumption_kwh"))\
#                          .otherwise(0))\
#               .withColumn("energy_to_get",when(col("consumption_kwh")-col("production_kwh")>0,col("consumption_kwh")-col("production_kwh"))\
#                          .otherwise(0))\
#               .withColumn("battery_charge_at_start",lit(0.00000))\
#               .withColumn("battery_charge_at_end",lit(0.00000))

# (optional) setting the starting state of the battery:
# basic_p.at[0,"battery_charge_start"] = 10

#calculating the amount we can store of the surplus generated
if basic_p.at[0,'energy_to_store'] > 0:
        temp = basic_p.at[0,'battery_charge_start'] + basic_p.at[0,'energy_to_store']
        basic_p.at[0,'battery_charge_end'] = __builtins__.min(temp,basic_p.at[0,'battery_charge_start'] + Charge_rate, Capacity)
       
            
#calculating how much of the deficit we can cover from our battery
else:
        temp = basic_p.at[0,'battery_charge_start']  - __builtins__.min(basic_p.at[0,'energy_to_get'], Discharge_rate)
        if temp > Capacity_min:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = Capacity_min

for i in range(1,int(basic_p.size/8)):
    basic_p.at[i,'battery_charge_start'] = basic_p.at[i-1,'battery_charge_end']
    if basic_p.at[i,'energy_to_store'] > 0:
        temp = basic_p.at[i,'battery_charge_start'] + basic_p.at[i,'energy_to_store']
        if temp <= Capacity:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = Capacity
    else:
        temp = basic_p.at[i,'battery_charge_start'] - __builtins__.min(basic_p.at[i,'energy_to_get'], Discharge_rate)
        if temp > Capacity_min:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = Capacity_min
#basic_p.tail(10)

# from the data previously calculated, we determine how much energy we used from our own storage and from the grid,
# furthermore how much energy we feed our system or the grid.
# last we calculate the energy price and the revenue. 
basic_p = basic_p.assign(taking_from_battery = lambda x: np.where(x['battery_charge_start'] - x['battery_charge_end'] > 0, x['battery_charge_start'] - x['battery_charge_end'], 0),
                       feeding_battery   = lambda x: np.where(x['battery_charge_end'] - x['battery_charge_start'] > 0, x['battery_charge_end'] - x['battery_charge_start'], 0),
                       taking_from_grid  = lambda x: x['energy_to_get']-x['taking_from_battery'],
                       feeding_grid      = lambda x: x['energy_to_store']-x['feeding_battery'],
                       price             = lambda x: (x['taking_from_grid']*x['price_eur_per_mwh']-x['feeding_grid']*x['price_eur_per_mwh'])*0.00025) 

basic_p.head(100)



,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,energy_to_store,energy_to_get,battery_charge_start,battery_charge_end,taking_from_battery,feeding_battery,taking_from_grid,feeding_grid,price
0,2022-01-01 00:00:00,0.18187,0.0,41.330002,0.0,0.18187,0.00000,0.00000,0.00000,0.0,0.18187,0.0,0.001879
1,2022-01-01 00:15:00,0.18158,0.0,41.330002,0.0,0.18158,0.00000,0.00000,0.00000,0.0,0.18158,0.0,0.001876
2,2022-01-01 00:30:00,0.18087,0.0,41.330002,0.0,0.18087,0.00000,0.00000,0.00000,0.0,0.18087,0.0,0.001869
3,2022-01-01 00:45:00,0.18103,0.0,41.330002,0.0,0.18103,0.00000,0.00000,0.00000,0.0,0.18103,0.0,0.001870
4,2022-01-01 01:00:00,0.18172,0.0,43.220001,0.0,0.18172,0.00000,0.00000,0.00000,0.0,0.18172,0.0,0.001963
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2022-01-01 23:45:00,0.18095,0.0,57.080002,0.0,0.18095,40.54918,40.36823,0.18095,0.0,0.00000,0.0,0.000000
96,2022-01-02 00:00:00,0.18187,0.0,52.590000,0.0,0.18187,40.36823,40.18636,0.18187,0.0,0.00000,0.0,0.000000
97,2022-01-02 00:15:00,0.18158,0.0,52.590000,0.0,0.18158,40.18636,40.00478,0.18158,0.0,0.00000,0.0,0.000000
98,2022-01-02 00:30:00,0.18087,0.0,52.590000,0.0,0.18087,40.00478,39.82391,0.18087,0.0,0.00000,0.0,0.000000


In [34]:
simple_total_generation = basic_p.assign(generation = lambda x: x['production_kwh'])['generation'].sum()
simple_total_export = basic_p["feeding_grid"].sum()
simple_self_consumption_rate = 1 - simple_total_export/simple_total_generation

simple_total_consumption = basic_p.assign(consumption = lambda x: x['consumption_kwh'])['consumption'].sum()
simple_total_import = basic_p['taking_from_grid'].sum()
simple_self_sufficiency_rate = 1 - simple_total_import/simple_total_consumption

simple_total_cost = basic_p[basic_p['price']>0]['price'].sum()
simple_total_revenue = basic_p[basic_p['price']<0]['price'].sum()*(-1)
simple_net_revenue = basic_p['price'].sum()*(-1)

print("Total generation: ",simple_total_generation, " kw")
print(f"Total consumption: {simple_total_consumption}")
print(f"Self consumption rate: {simple_self_consumption_rate}")
print(f"Self sufficiency rate: {simple_self_sufficiency_rate}")
print(f"Total cost: {simple_total_cost}")
print(f"Total revenue: {simple_total_revenue}")
print(f"Net revenue: {simple_net_revenue}")

Total generation:  142054.31  kw
Total consumption: 19999.2734375
Self consumption rate: 0.1408310962443271
Self sufficiency rate: 0.9955727308185683
Total cost: 11.052347648318792
Total revenue: 5766.843578815964
Net revenue: 5755.791231167644


# Rule Based Strategy

If there is a deficit, we check whether the energy price reaches the current upper price threshold. If not we buy from the grid, if yes then we use the backup stored in BESS, if we can't cover demand from BESS, then power is bought from the grid.

For surplus, we load as much energy as we can into BESS, the remainder is sold to the grid.

In both cases we check whether the energy price is below the lower price threshold, then we charge the battery as much as we can from the grid.

The upper threshold is calculated from the upper (.25,.5,.75) percentile of the daily forcasted price data.
The lower threshold is calculated from the lower .25 percentile of the previous 15 days' price data.

both thresholds are recalculated daily.

In [132]:
#function for determining season
def get_season(timestamp):
    # Alakítsuk át a timestamp-et datetime objektummá
    month = timestamp.month
    day = timestamp.day
    
    if (month == 12 and day >= 21) or (1 <= month <= 2) or (month == 3 and day <= 20):
        return 'Winter'
    elif (month == 3 and day >= 21) or (4 <= month <= 5) or (month == 6 and day <= 20):
        return 'Spring'
    elif (month == 6 and day >= 21) or (7 <= month <= 8) or (month == 9 and day <= 21):
        return 'Summer'
    else:
        return 'Autumn'

season_udf = udf(get_season, StringType())

In [68]:
get_season(rule_based.at[0,"timestamp"])

'Winter'

In [69]:
percentiles = {'Winter':0.75,'Spring':0.5,'Summer':0.25,'Autumn':0.5}

rule_based = process.assign(Surplus = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                            Deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       Buy_price     = float(0.00000),
                       Selling_price = float(0.0000))
n_ut = 1
n_lt = 1

#first row
UT = rule_based.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation='higher')
LT = 0

if rule_based.at[0,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
        
        # cheaper than UT
        if rule_based.at[0,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < Capacity):
               rule_based.at[0,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[0,"SOC_start"],Charge_rate])
                #else P_G2B = 0
        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based.at[0,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"Deficit"], rule_based.at[0,"SOC_start"]- Capacity_min, Discharge_rate])
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"] - rule_based.at[0,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]
                
    # Surplus side            
else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]

        # is BESS full?
        if rule_based.at[0,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based.at[0,"Surplus"] < Charge_rate:
                
                rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"Surplus"], Capacity - rule_based.at[0,"SOC_start"])
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                
                if rule_based.at[0,"price_eur_per_mwh"] < LT:
                    rule_based.at[0,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[0,"P_P2B"], Capacity - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
                    # else P_G2B = 0
            
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based.at[0,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[0,"SOC_start"])
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"]

#P_P2L

#P_P2G
#P_G2B
#P_G2L


rule_based.at[0,"SOC_end"] = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
rule_based.at[0,"Buy_price"] = (rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"])*rule_based.at[0,"price_eur_per_mwh"]*0.00025
rule_based.at[0,"Selling_price"] = rule_based.at[0,"P_P2G"]*rule_based.at[0,"price_eur_per_mwh"]*0.00025

for i in range(1,int(rule_based.size/16)):
    # setting SOC 
    rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based.size/16):
        UT = rule_based.iloc[i : i + 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation='higher')
    if (n_lt % 96) & (n_lt >= 1440):
        LT = rule_based.iloc[i - 1439 : i]['price_eur_per_mwh'].quantile(0.05, interpolation='lower')
    n_ut += 1
    n_lt += 1

    # comparing demand and production
    # Deficit side
    if rule_based.at[i,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
        
        # cheaper than UT
        if rule_based.at[i,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < Capacity):
               rule_based.at[i,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[i,"SOC_start"],Charge_rate])
                #else P_G2B = 0
        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based.at[i,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min, Discharge_rate])
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"] - rule_based.at[i,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]
                
    # Surplus side            
    else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]

        # is BESS full?
        if rule_based.at[i,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based.at[i,"Surplus"] < Charge_rate:
                
                rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"Surplus"], Capacity - rule_based.at[i,"SOC_start"])
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                
                if rule_based.at[i,"price_eur_per_mwh"] < LT:
                    rule_based.at[i,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[i,"P_P2B"], Capacity - rule_based.at[i,"SOC_start"] - rule_based.at[i,"P_P2B"])
                    # else P_G2B = 0
            
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based.at[i,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[i,"SOC_start"])
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"]

    #executing the calculations
    rule_based.at[i,"SOC_end"]       = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
    rule_based.at[i,"Buy_price"]     = (rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"])*rule_based.at[i,"price_eur_per_mwh"]*0.00025
    rule_based.at[i,"Selling_price"] = rule_based.at[i,"P_P2G"]*rule_based.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0
LT=0


In [70]:
rule_total_generation = rule_based["production_kwh"].sum()
rule_total_export = rule_based["P_P2G"].sum()
rule_self_consumption_rate = 1 - rule_total_export/rule_total_generation

rule_total_consumption = rule_based["consumption_kwh"].sum()
rule_total_import = rule_based["P_G2L"].sum() + rule_based["P_G2B"].sum()
rule_self_sufficiency_rate = 1 - rule_total_import/rule_total_consumption

rule_total_cost = rule_based["Buy_price"].sum()
rule_total_revenue = rule_based["Selling_price"].sum()
rule_net_revenue = rule_total_revenue - rule_total_cost

print("Total generation: ",rule_total_generation, " kw")
print(f"Total consumption: {rule_total_consumption}")
print(f"Self consumption rate: {rule_self_consumption_rate}")
print(f"Self sufficiency rate: {rule_self_sufficiency_rate}")
print(f"Total cost: {rule_total_cost}")
print(f"Total revenue: {rule_total_revenue}")
print(f"Net revenue: {rule_net_revenue}")

Total generation:  142054.31  kw
Total consumption: 19999.2734375
Self consumption rate: 0.10997077827071278
Self sufficiency rate: 0.7762112366639278
Total cost: 165.7640695553988
Total revenue: 5957.666323946962
Net revenue: 5791.902254391563


# Rule Based Strategy with no LT

same strategy with one key difference: no lower threshold is used to buy cheaper energy.

In [71]:
rule_based_no_lt = process.assign(Surplus = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, x['production_kwh'] - x['consumption_kwh'], 0),                       Deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, x['consumption_kwh'] - x['production_kwh'], 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       Buy_price     = float(0.00000),
                       Selling_price = float(0.0000))
n_ut = 1


#first row
UT = rule_based_no_lt.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation='higher')


if rule_based_no_lt.at[0,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based_no_lt.at[0,"P_P2L"] = rule_based_no_lt.at[0,"production_kwh"]
        
        # cheaper than UT
        if rule_based_no_lt.at[0,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"Deficit"]

        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based_no_lt.at[0,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based_no_lt.at[0,"P_B2L"] = __builtins__.min([rule_based_no_lt.at[0,"Deficit"], rule_based_no_lt.at[0,"SOC_start"]- Capacity_min, Discharge_rate])
                rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"Deficit"] - rule_based_no_lt.at[0,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"Deficit"]
                
    # Surplus side            
else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based_no_lt.at[0,"P_P2L"] = rule_based_no_lt.at[0,"consumption_kwh"]

        # is BESS full?
        if rule_based_no_lt.at[0,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based_no_lt.at[0,"Surplus"] < Charge_rate:
                
                rule_based_no_lt.at[0,"P_P2B"] = __builtins__.min(rule_based_no_lt.at[0,"Surplus"], Capacity - rule_based_no_lt.at[0,"SOC_start"])
                rule_based_no_lt.at[0,"P_P2G"] = rule_based_no_lt.at[0,"Surplus"] - rule_based_no_lt.at[0,"P_P2B"]
                
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based_no_lt.at[0,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based_no_lt.at[0,"SOC_start"])
                rule_based_no_lt.at[0,"P_P2G"] = rule_based_no_lt.at[0,"Surplus"] - rule_based_no_lt.at[0,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"]

#P_P2L

#P_P2G
#P_G2B
#P_G2L


rule_based_no_lt.at[0,"SOC_end"] = rule_based_no_lt.at[0,"SOC_start"] + rule_based_no_lt.at[0,"P_P2B"] + rule_based_no_lt.at[0,"P_G2B"] - rule_based_no_lt.at[0,"P_B2L"]
rule_based_no_lt.at[0,"Buy_price"] = (rule_based_no_lt.at[0,"P_G2B"] + rule_based_no_lt.at[0,"P_G2L"])*rule_based_no_lt.at[0,"price_eur_per_mwh"]*0.00025
rule_based_no_lt.at[0,"Selling_price"] = rule_based_no_lt.at[0,"P_P2G"]*rule_based_no_lt.at[0,"price_eur_per_mwh"]*0.00025

for i in range(1,int(rule_based_no_lt.size/16)):
    # setting SOC 
    rule_based_no_lt.at[i,"SOC_start"] = rule_based_no_lt.at[i-1,"SOC_end"]
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based_no_lt.size/16):
        UT = rule_based_no_lt.iloc[i : i + 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation='higher')
    
    n_ut += 1


    # comparing demand and production
    # Deficit side
    if rule_based_no_lt.at[i,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based_no_lt.at[i,"P_P2L"] = rule_based_no_lt.at[i,"production_kwh"]
        
        # cheaper than UT
        if rule_based_no_lt.at[i,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"Deficit"]

            #cheaper than LT and there is space
            
        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based_no_lt.at[i,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based_no_lt.at[i,"P_B2L"] = __builtins__.min([rule_based_no_lt.at[i,"Deficit"], rule_based_no_lt.at[i,"SOC_start"]- Capacity_min, Discharge_rate])
                rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"Deficit"] - rule_based_no_lt.at[i,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"Deficit"]
                
    # Surplus side            
    else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based_no_lt.at[i,"P_P2L"] = rule_based_no_lt.at[i,"consumption_kwh"]

        # is BESS full?
        if rule_based_no_lt.at[i,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based_no_lt.at[i,"Surplus"] < Charge_rate:
                
                rule_based_no_lt.at[i,"P_P2B"] = __builtins__.min(rule_based_no_lt.at[i,"Surplus"], Capacity - rule_based_no_lt.at[i,"SOC_start"])
                rule_based_no_lt.at[i,"P_P2G"] = rule_based_no_lt.at[i,"Surplus"] - rule_based_no_lt.at[i,"P_P2B"]
                
            
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based_no_lt.at[i,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based_no_lt.at[i,"SOC_start"])
                rule_based_no_lt.at[i,"P_P2G"] = rule_based_no_lt.at[i,"Surplus"] - rule_based_no_lt.at[i,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based_no_lt.at[i,"P_P2G"] = rule_based_no_lt.at[i,"Surplus"]

    #executing the calculations
    rule_based_no_lt.at[i,"SOC_end"]       = rule_based_no_lt.at[i,"SOC_start"] + rule_based_no_lt.at[i,"P_P2B"] + rule_based_no_lt.at[i,"P_G2B"] - rule_based_no_lt.at[i,"P_B2L"]
    rule_based_no_lt.at[i,"Buy_price"]     = (rule_based_no_lt.at[i,"P_G2B"] + rule_based_no_lt.at[i,"P_G2L"])*rule_based_no_lt.at[i,"price_eur_per_mwh"]*0.00025
    rule_based_no_lt.at[i,"Selling_price"] = rule_based_no_lt.at[i,"P_P2G"]*rule_based_no_lt.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0



In [72]:
rule_no_lt_total_generation = rule_based_no_lt["production_kwh"].sum()
rule_no_lt_total_export = rule_based_no_lt["P_P2G"].sum()
rule_no_lt_self_consumption_rate = 1 - rule_no_lt_total_export/rule_no_lt_total_generation

rule_no_lt_total_consumption = rule_based_no_lt["consumption_kwh"].sum()
rule_no_lt_total_import = rule_based_no_lt["P_G2L"].sum() + rule_based_no_lt["P_G2B"].sum()
rule_no_lt_self_sufficiency_rate = 1 - rule_no_lt_total_import/rule_no_lt_total_consumption

rule_no_lt_total_cost = rule_based_no_lt["Buy_price"].sum()
rule_no_lt_total_revenue = rule_based_no_lt["Selling_price"].sum()
rule_no_lt_net_revenue = rule_no_lt_total_revenue - rule_no_lt_total_cost

print("Total generation: ",rule_no_lt_total_generation, " kw")
print(f"Total consumption: {rule_no_lt_total_consumption}")
print(f"Self consumption rate: {rule_no_lt_self_consumption_rate}")
print(f"Self sufficiency rate: {rule_no_lt_self_sufficiency_rate}")
print(f"Total cost: {rule_no_lt_total_cost}")
print(f"Total revenue: {rule_no_lt_total_revenue}")
print(f"Net revenue: {rule_no_lt_net_revenue}")

Total generation:  142054.31  kw
Total consumption: 19999.2734375
Self consumption rate: 0.11425718004489493
Self sufficiency rate: 0.806657435573759
Total cost: 152.39337612944672
Total revenue: 5934.318386659816
Net revenue: 5781.925010530369


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
70071,2023-12-31 21:45:00,0.15386,0.0,10.68
70072,2023-12-31 22:00:00,0.15380,0.0,14.95
70073,2023-12-31 22:15:00,0.15365,0.0,14.95
70074,2023-12-31 22:30:00,0.15369,0.0,14.95
70075,2023-12-31 22:45:00,0.15634,0.0,14.95


# Experiment for rule based hyper parametering

getting seasonal data:

In [103]:
seasonal_data = production.withColumn("season",season_udf(col("timestamp"))).withColumn("week", weekofyear(col("timestamp"))).withColumn("year",year(col("timestamp")))
avg_data= seasonal_data.groupBy("season").agg(avg("production_kwh").alias("avg_production"))
seasonal_data = seasonal_data.join(avg_data,"season","inner")\
                             .groupBy("season","week").agg(avg("production_kwh").alias("weekly_avg"),max("avg_production").alias("seasonal_avg"),max("year").alias("year"))\
                             .withColumn("diff",abs(col("seasonal_avg")-col("weekly_avg")))
window_spec = Window.partitionBy("season").orderBy(col("diff"))

closest_weeks = seasonal_data.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")

closest_weeks.select("season","year","week","seasonal_avg","weekly_avg","diff").show()


+------+----+----+------------------+------------------+--------------------+
|season|year|week|      seasonal_avg|        weekly_avg|                diff|
+------+----+----+------------------+------------------+--------------------+
|Autumn|2023|  45| 1.060399313461649|0.9795070703125004| 0.08089224314914856|
|Spring|2023|  17| 2.762451384593052|2.8057818705357165| 0.04333048594266442|
|Summer|2023|  36|2.8816566322244705|2.8859151849888427|0.004258552764372148|
|Winter|2023|   5|1.3595721927879036|1.2863808284040175| 0.07319136438388618|
+------+----+----+------------------+------------------+--------------------+



extracting the chosen weeks according to the results:

In [130]:
test_data = spark.createDataFrame(process).withColumn("season",season_udf(col("timestamp")))\
                      .withColumn("week", weekofyear(col("timestamp")))\
                      .withColumn("year",year(col("timestamp")))\
                   .filter((col("year") == 2023) & ((col("week") == 5) | (col("week") == 17) | (col("week") == 36) | (col("week") == 45)))
test_data = test_data.toPandas()
test_data.head()



24/06/07 17:19:54 WARN TaskSetManager: Stage 352 contains a task of very large size (1338 KiB). The maximum recommended task size is 1000 KiB.


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,season,week,year
0,2023-01-30 00:00:00,0.18221,0.0,79.129997,Winter,5,2023
1,2023-01-30 00:15:00,0.18426,0.0,79.129997,Winter,5,2023
2,2023-01-30 00:30:00,0.18654,0.0,79.129997,Winter,5,2023
3,2023-01-30 00:45:00,0.18693,0.0,79.129997,Winter,5,2023
4,2023-01-30 01:00:00,0.18698,0.0,52.790001,Winter,5,2023


testing the different percentile variations:

In [144]:
values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
variations = list(itertools.product(values,repeat=4))


0.1, 0.2, 0.2, 0.1


In [154]:
dataset = []
columns = ["percentiles", "self_consumption_rate","self_sufficiency_rate","total_cost","total_revenue","net_revenue"]
variations = [(0.1,0.2,0.2,0.1)]
for v in variations:
    percentiles['Winter'] = v[0]
    percentiles['Spring'] = v[1]
    percentiles['Summer'] = v[2]
    percentiles['Autumn'] = v[3]
    rule_based = test_data.assign(Surplus = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                            Deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       Buy_price     = float(0.00000),
                       Selling_price = float(0.0000))
    n_ut = 1
    n_lt = 1
    
    #first row
    UT = rule_based.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation='higher')
    LT = 0
    
    if rule_based.at[0,"Deficit"] > 0:
            #P_P2B = 0
            #P_P2G = 0
            rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
            
            # cheaper than UT
            if rule_based.at[0,"price_eur_per_mwh"] < UT:
                #P_B2L = 0
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]
    
                #cheaper than LT and there is space
                if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < Capacity):
                   rule_based.at[0,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[0,"SOC_start"],Charge_rate])
                    #else P_G2B = 0
            # expensive
            else:
                # P_G2B = 0
    
                # BESS not empty
                if rule_based.at[0,"SOC_start"] > Capacity_min:
    
                    #if rule_based.at[i,"Deficit"] < Discharge_rate:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                    #
                    #else:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                    rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"Deficit"], rule_based.at[0,"SOC_start"]- Capacity_min, Discharge_rate])
                    rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"] - rule_based.at[0,"P_B2L"]
                #BESS is empty
                else:
                    #P_B2L = 0
                    rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]
                    
        # Surplus side            
    else:
            #P_G2L = 0
            #P_B2L = 0
            rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]
    
            # is BESS full?
            if rule_based.at[0,"SOC_start"] < Capacity:
    
                # Is surplus < Charge rate
                if rule_based.at[0,"Surplus"] < Charge_rate:
                    
                    rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"Surplus"], Capacity - rule_based.at[0,"SOC_start"])
                    rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                    
                    if rule_based.at[0,"price_eur_per_mwh"] < LT:
                        rule_based.at[0,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[0,"P_P2B"], Capacity - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
                        # else P_G2B = 0
                
                # surplus bigger
                else:
                    # P_G2B = 0
                    rule_based.at[0,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[0,"SOC_start"])
                    rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                    
            # BESS is full
            else:
                #P_P2B = 0
                #P_G2B = 0
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"]
    
    #P_P2L
    
    #P_P2G
    #P_G2B
    #P_G2L
    
    
    rule_based.at[0,"SOC_end"] = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
    rule_based.at[0,"Buy_price"] = (rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"])*rule_based.at[0,"price_eur_per_mwh"]*0.00025
    rule_based.at[0,"Selling_price"] = rule_based.at[0,"P_P2G"]*rule_based.at[0,"price_eur_per_mwh"]*0.00025
    
    for i in range(1,int(rule_based.size/16)):
        # setting SOC 
        rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
        # calculating thresholds
        if (n_ut % 96 == 0) & (n_ut != rule_based.size/16):
            UT = rule_based.iloc[i : i + 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation='higher')
        if (n_lt % 96) & (n_lt >= 1440):
            LT = rule_based.iloc[i - 1439 : i]['price_eur_per_mwh'].quantile(0.05, interpolation='lower')
        n_ut += 1
        n_lt += 1
    
        # comparing demand and production
        # Deficit side
        if rule_based.at[i,"Deficit"] > 0:
            #P_P2B = 0
            #P_P2G = 0
            rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
            
            # cheaper than UT
            if rule_based.at[i,"price_eur_per_mwh"] < UT:
                #P_B2L = 0
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]
    
                #cheaper than LT and there is space
                if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < Capacity):
                   rule_based.at[i,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[i,"SOC_start"],Charge_rate])
                    #else P_G2B = 0
            # expensive
            else:
                # P_G2B = 0
    
                # BESS not empty
                if rule_based.at[i,"SOC_start"] > Capacity_min:
    
                    #if rule_based.at[i,"Deficit"] < Discharge_rate:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                    #
                    #else:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min, Discharge_rate])
                    rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"] - rule_based.at[i,"P_B2L"]
                #BESS is empty
                else:
                    #P_B2L = 0
                    rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]
                    
        # Surplus side            
        else:
            #P_G2L = 0
            #P_B2L = 0
            rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]
    
            # is BESS full?
            if rule_based.at[i,"SOC_start"] < Capacity:
    
                # Is surplus < Charge rate
                if rule_based.at[i,"Surplus"] < Charge_rate:
                    
                    rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"Surplus"], Capacity - rule_based.at[i,"SOC_start"])
                    rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                    
                    if rule_based.at[i,"price_eur_per_mwh"] < LT:
                        rule_based.at[i,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[i,"P_P2B"], Capacity - rule_based.at[i,"SOC_start"] - rule_based.at[i,"P_P2B"])
                        # else P_G2B = 0
                
                # surplus bigger
                else:
                    # P_G2B = 0
                    rule_based.at[i,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[i,"SOC_start"])
                    rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                    
            # BESS is full
            else:
                #P_P2B = 0
                #P_G2B = 0
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"]
    
        #executing the calculations
        rule_based.at[i,"SOC_end"]       = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
        rule_based.at[i,"Buy_price"]     = (rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"])*rule_based.at[i,"price_eur_per_mwh"]*0.00025
        rule_based.at[i,"Selling_price"] = rule_based.at[i,"P_P2G"]*rule_based.at[i,"price_eur_per_mwh"]*0.00025
                
    UT=0
    LT=0
    r_total_generation = rule_based["production_kwh"].sum()
    r_total_export = rule_based["P_P2G"].sum()
    r_self_consumption_rate = 1 - r_total_export/r_total_generation
    
    r_total_consumption = rule_based["consumption_kwh"].sum()
    r_total_import = rule_based["P_G2L"].sum() + rule_based["P_G2B"].sum()
    r_self_sufficiency_rate = 1 - r_total_import/r_total_consumption
    
    r_total_cost = rule_based["Buy_price"].sum()
    r_total_revenue = rule_based["Selling_price"].sum()
    r_net_revenue = r_total_revenue - r_total_cost
    
    string_percentiles = ','.join([str(value) for value in percentiles.values()])
    dataset.append((string_percentiles, float(r_self_consumption_rate), float(r_self_sufficiency_rate), float(r_total_cost), float(r_total_revenue), float(r_net_revenue)))
    
    print(percentiles.values())
    print(f"Self consumption rate: {r_self_consumption_rate}")
    print(f"Self sufficiency rate: {r_self_sufficiency_rate}")
    print(f"Total cost: {r_total_cost}")
    print(f"Total revenue: {r_total_revenue}")
    print(f"Net revenue: {r_net_revenue}")

df = spark.createDataFrame(dataset, columns)
df.show()

dict_values([0.1, 0.2, 0.2, 0.1])
Self consumption rate: 0.17136233326447914
Self sufficiency rate: 0.959219946181709
Total cost: 0.9456301695123548
Total revenue: 95.2096907537181
Net revenue: 94.26406058420575
+---------------+---------------------+---------------------+------------------+----------------+-----------------+
|    percentiles|self_consumption_rate|self_sufficiency_rate|        total_cost|   total_revenue|      net_revenue|
+---------------+---------------------+---------------------+------------------+----------------+-----------------+
|0.1,0.2,0.2,0.1|  0.17136233326447914|    0.959219946181709|0.9456301695123548|95.2096907537181|94.26406058420575|
+---------------+---------------------+---------------------+------------------+----------------+-----------------+



In [158]:
df = df.repartition(1)
df.write.format("csv").option("header","true").option("delimiter",";").mode("overwrite").save("../output/percentiles.csv")